# Import and Shared functions

In [ ]:
import os, sys, re, ast
from glob import glob
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

AXES   = ["M1", "M2", "M3"]        # Base-frame motor axes -- shared by every θ/ω field in both KFLOG and PIDLOG
TOPO   = ["Az", "Alt", "Roll"]     # Topocentric frame (α_*) axes
EQU    = ["RA", "Dec", "PA"]       # Equatorial frame (Δ_*) axes -- where issue #88 shows up
ARCSEC = 3600                      # deg -> arcsec, for readability at the scale this investigation cares about

def wrap_deg(x):
    """Wrap a degree value/difference into (-180, 180]. Every 'error' plot below is pv - sp,
    which needs this: without it, sp=359.9/pv=0.1 (e.g. Az crossing 0/360, or an RA/PA axis
    wrapping) gives a spurious -359.8 instead of the true +0.2."""
    return (x + 180) % 360 - 180

def parse_val(v):
    v = v.strip()
    try:
        return float(v)
    except ValueError:
        return v

def parse_payload_line(line, tag):
    """Split 'TIMESTAMP INFO <TAG> {dict}' and literal_eval the trailing dict."""
    if f" {tag} " not in line:
        return None
    ts, _, body = line.partition(f" {tag} ")
    ts = ts.split(" INFO")[0].strip()
    try:
        payload = ast.literal_eval(body.strip())
    except (ValueError, SyntaxError):
        return None
    rec = {"timestamp": ts}
    for key, val in payload.items():
        if isinstance(val, list):
            for i, v in enumerate(val):
                # None (e.g. PECLOG's guide field on a cycle where only one axis updated) must
                # not be stringified to "None" first -- float("None") raises, so parse_val would
                # otherwise leave the literal string "None" sitting in a numeric column.
                rec[f"{key}_{i+1}"] = float("nan") if v is None else parse_val(str(v))
        else:
            rec[key] = float("nan") if val is None else parse_val(str(val))
    return rec

def resolve_log_files(pattern):
    """
    Expand a glob pattern (or accept an explicit list) into the set of rotated driver
    logs to load. A multi-hour capture routinely spans several rotated files regardless
    of the configured per-file size cap, so this is the normal path, not a special case.
    Only matches 'alpaca.log' and 'alpaca.log.<N>' -- excludes renamed/archived variants
    (e.g. 'alpaca.mark_*.log') and stray ':Zone.Identifier' sidecar files.
    """
    if isinstance(pattern, (list, tuple)):
        return list(pattern)
    candidates = glob(pattern)
    return [p for p in candidates if re.search(r"alpaca\.log(\.\d+)?$", os.path.basename(p))]

def load_kf_pid(log_paths):
    """
    Parse KFLOG and PIDLOG lines from one or more rotated driver logs
    (Config.log_position = true) into two DataFrames. Accepts a single path, a glob
    pattern (e.g. '../logs/alpaca.log*'), or an explicit list of paths -- all rows are
    concatenated and re-sorted by timestamp, so file order/naming doesn't matter.
    """
    paths = resolve_log_files(log_paths) if isinstance(log_paths, str) else list(log_paths)
    if not paths:
        raise FileNotFoundError(f"No log files matched: {log_paths!r}")
    kf_rows, pid_rows = [], []
    for log_path in paths:
        if not os.path.exists(log_path):
            raise FileNotFoundError(f"log_path does not exist: {log_path!r}")
        # encoding='utf-8' is required: the driver writes θ/ω/Δ/α as UTF-8 (log.py's
        # RotatingFileHandler is pinned to utf-8), but open() without an explicit encoding
        # falls back to the OS default -- cp1252 on Windows -- which silently mangles those
        # keys instead of raising, so columns like "θ_sp_1" quietly never get created.
        with open(log_path, encoding="utf-8") as f:
            for line in f:
                if " KFLOG " in line:
                    rec = parse_payload_line(line, "KFLOG")
                    if rec: kf_rows.append(rec)
                elif " PIDLOG " in line:
                    rec = parse_payload_line(line, "PIDLOG")
                    if rec: pid_rows.append(rec)

    if not kf_rows and not pid_rows:
        raise ValueError(f"No KFLOG/PIDLOG lines found in {paths!r} -- was Config.log_position true during this session?")

    def finalize(rows):
        df = pd.DataFrame(rows)
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        # Dedup on the FULL row, not just timestamp -- KFLOG can log two genuinely distinct
        # messages (e.g. a delayed one immediately followed by a fresh one) that round to the
        # same millisecond string. Deduping on timestamp alone silently discards one of them,
        # which corrupts exactly the batch/near-duplicate timing this notebook exists to study.
        # A true duplicate (e.g. from overlapping rotated log segments) still has identical
        # values in every column and is still caught here.
        df = df.sort_values("timestamp", kind="stable").drop_duplicates().reset_index(drop=True)
        df["t_sec"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()
        df["gap_sec"] = df["timestamp"].diff().dt.total_seconds()
        return df

    kf_df  = finalize(kf_rows)  if kf_rows  else pd.DataFrame()
    pid_df = finalize(pid_rows) if pid_rows else pd.DataFrame()
    return kf_df, pid_df


# Load data

In [ ]:
# ── Choose log path (last set log_path is what is used) ────────────────────────
# Single file:            '../logs/alpaca.log'
# All rotated files:      '../logs/alpaca.log*'   (glob -- handles a multi-hour run spanning several files)
# Explicit file list:     ['../logs/alpaca.log.2', '../logs/alpaca.log.1', '../logs/alpaca.log']
log_path = ['../logs/logs/alpaca.soak_nopec_Beta4.3_08_27a.log', '../logs/logs/alpaca.soak_nopec_Beta4.3_08_27b.log']
log_path = ['../logs/logs/alpaca.soak_nopec_Beta4.3_08_28a.log']
log_path = ['../logs/logs/alpaca.soak_nopec_Beta4.3_08_28k1.log','../logs/logs/alpaca.soak_nopec_Beta4.3_08_28k2.log','../logs/logs/alpaca.soak_nopec_Beta4.3_08_28k3.log','../logs/logs/alpaca.soak_nopec_Beta4.3_08_28k4.log']
log_path = ['../logs/logs/alpaca.soak_nopec_Beta4.3_08_29a.log']
log_path = ['../logs/logs/alpaca.soak_nopec_Beta4.3_08_29b1.log','../logs/logs/alpaca.soak_nopec_Beta4.3_08_29b2.log']


resolved_files = resolve_log_files(log_path) if isinstance(log_path, str) else list(log_path)
print(f"Loading {len(resolved_files)} file(s):")
for p in resolved_files:
    print(f"  {p}  ({os.path.getsize(p)/1e6:.1f} MB)")
print()

kf_df, pid_df = load_kf_pid(log_path)
print(f"KF  samples: {len(kf_df)}"  + (f"  ({kf_df.t_sec.iloc[-1]/60:.1f} min span)"  if len(kf_df)  else ""))
print(f"PID samples: {len(pid_df)}" + (f"  ({pid_df.t_sec.iloc[-1]/60:.1f} min span)" if len(pid_df) else ""))
if len(kf_df):
    print(f"KF  time range: {kf_df.timestamp.iloc[0]}  ->  {kf_df.timestamp.iloc[-1]}")
if len(pid_df):
    print(f"PID time range: {pid_df.timestamp.iloc[0]}  ->  {pid_df.timestamp.iloc[-1]}")
print()
print("KF columns:", list(kf_df.columns))
print("PID columns:", list(pid_df.columns))


# Exposure subgrouping (from PECLOG)

`PECLOG` (parsed the same way as `analyse_pec.ipynb`) is written once per guide-sync cycle
(`control.py:_pec_log()`, called from `process_guide_sync()`) -- it marks a sync-guide event,
**not** one-per-exposure. If consecutive PECLOG entries are further apart than `EXPOSURE_SEC`,
that gap is several back-to-back exposures, not one long idle gap -- e.g. two PECLOGs 140s
apart at the 30s default means one sync-guide followed by ~4 exposures before the next sync.
This tags every PID/KF tick with which of those back-to-back exposures it falls in (and where
within that exposure), so anomalies can be grouped/filtered by exposure -- e.g. "which
sub-exposures had a flagged RA/PA event during them" -- rather than only viewed against the
whole session. Optional: this capture only has PECLOG entries if guiding was active during it.

In [ ]:
EXPOSURE_SEC = 30   # configurable -- assumed exposure duration; exposures are assumed to run
                     # back-to-back starting shortly after each PECLOG (see markdown above)

def load_peclog(log_paths):
    """
    Parse PECLOG lines using the same dict-payload convention as KFLOG/PIDLOG (see
    parse_payload_line above) -- control.py's _pec_log() emits one dict per guide-sync
    cycle. Returns an empty DataFrame (not an error) when none are found -- guiding/exposing
    isn't active in every capture, so this is optional/supplementary.
    """
    paths = resolve_log_files(log_paths) if isinstance(log_paths, str) else list(log_paths)
    rows = []
    for log_path in paths:
        with open(log_path, encoding="utf-8") as f:
            for line in f:
                if " PECLOG " not in line:
                    continue
                rec = parse_payload_line(line, "PECLOG")
                if rec: rows.append(rec)
    df = pd.DataFrame(rows)
    if len(df):
        df["timestamp"] = pd.to_datetime(df["timestamp"])
        df = df.drop_duplicates(subset="timestamp").sort_values("timestamp").reset_index(drop=True)
    return df

peclog_df = load_peclog(log_path)

if len(peclog_df) and len(pid_df):
    origin = kf_df.timestamp.iloc[0] if len(kf_df) else pid_df.timestamp.iloc[0]
    peclog_df["t_sec"] = (peclog_df.timestamp - origin).dt.total_seconds()

    guide_starts = peclog_df.t_sec.to_numpy()
    t = pid_df.t_sec.to_numpy()
    guide_block = np.searchsorted(guide_starts, t, side="right") - 1   # which PECLOG-to-PECLOG interval this tick is in
    has_block = guide_block >= 0
    elapsed = np.full(len(pid_df), np.nan)                             # seconds since the most recent PECLOG
    elapsed[has_block] = t[has_block] - guide_starts[guide_block[has_block]]

    # Consecutive EXPOSURE_SEC windows repeat back-to-back within each guide interval, not
    # just a single window right after the PECLOG -- see markdown above.
    exposure_num = np.full(len(pid_df), -1, dtype=int)
    exposure_num[has_block] = np.floor(elapsed[has_block] / EXPOSURE_SEC).astype(int)
    exposure_sec = np.where(has_block, elapsed - exposure_num * EXPOSURE_SEC, np.nan)  # position within that specific exposure

    pid_df["guide_block"]  = guide_block                                          # -1 = before the first PECLOG (no exposure info yet)
    pid_df["exposure_num"] = exposure_num                                         # which back-to-back exposure within this guide interval (0-indexed)
    pid_df["exposure_sec"] = exposure_sec                                         # elapsed seconds within that specific exposure
    pid_df["exposure_id"]  = np.where(has_block, guide_block * 100000 + exposure_num, -1)  # unique id across the whole run, for groupby

    n_exposures = pid_df.loc[pid_df.exposure_id >= 0, "exposure_id"].nunique()
    print(f"PECLOG entries: {len(peclog_df)}")
    print(f"Tagged {has_block.sum()} of {len(pid_df)} PID ticks ({has_block.mean()*100:.1f}%) across "
          f"{n_exposures} inferred {EXPOSURE_SEC:.0f}s exposure(s) spanning {len(peclog_df)} guide interval(s).")
else:
    pid_df["guide_block"]  = -1
    pid_df["exposure_num"] = -1
    pid_df["exposure_id"]  = -1
    pid_df["exposure_sec"] = np.nan
    print(f"No PECLOG entries in this capture -- exposure subgrouping unavailable for this run "
          f"(default exposure length is {EXPOSURE_SEC}s; re-run once a capture with active guiding is loaded).")


# Telemetry gaps (known 518-dropout artifact)

Per `docs/control.md`: the Polaris sometimes stops sending 518 telemetry for a few seconds,
during which `θ_pv` freezes while `θ_sp` keeps advancing -- producing a large but spurious
error spike once telemetry resumes, that looks like a control anomaly but isn't one. Flag
any gap much longer than the normal ~0.1-0.2s KFLOG cadence, before any plotting below, so
a brief total dropout can be masked out of every chart instead of blowing out its y-axis and
hiding the real, subtler variation those charts exist to show.

In [ ]:
GAP_THRESHOLD_SEC = 1.0   # normal cadence is ~0.1-0.2s; the known artifact runs a few seconds

if len(kf_df):
    gaps_df = kf_df[kf_df.gap_sec > GAP_THRESHOLD_SEC][["timestamp", "t_sec", "gap_sec"]].reset_index(drop=True)
    print(f"Flagged {len(gaps_df)} telemetry gap(s) > {GAP_THRESHOLD_SEC}s, "
          f"totalling {gaps_df.gap_sec.sum():.1f}s of lost telemetry")
else:
    gaps_df = pd.DataFrame(columns=["timestamp", "t_sec", "gap_sec"])
    print("No KF data loaded.")
gaps_df


# Gap masking (applies to every plot below)

Tags every KF/PID tick within a window around a flagged gap as `near_gap`. Every chart below
plots `series.mask(near_gap)` rather than the raw series -- pandas turns those ticks to NaN,
so plotly draws a visible break in the line instead of connecting through it or including it
in the y-axis autorange. The window covers the gap's *entire actual duration* (from
`gaps_df.gap_sec`, not a fixed constant -- some dropouts run over 100s, not a few seconds)
plus a small pad before it and a longer tail after it, since the KF/PID visibly takes tens of
seconds to resettle once telemetry resumes -- calibrated against a real run, widen/narrow
`GAP_EXCLUDE_BEFORE`/`GAP_EXCLUDE_AFTER` as needed.

In [ ]:
GAP_EXCLUDE_BEFORE = 2.0   # extra seconds before the last good sample to mask, as a safety pad
GAP_EXCLUDE_AFTER  = 30.0  # seconds after a gap to mask -- KF/PID visibly takes this long to resettle

def mask_near_gap(df, gaps_df, before_pad=GAP_EXCLUDE_BEFORE, after=GAP_EXCLUDE_AFTER):
    near = pd.Series(False, index=df.index)
    for _, g in gaps_df.iterrows():
        start = g.t_sec - g.gap_sec - before_pad   # back to the actual last good sample, not a fixed offset --
        end   = g.t_sec + after                     # a dropout can run well over 100s, not just a few
        near |= (df.t_sec >= start) & (df.t_sec <= end)
    return near

kf_df["near_gap"]  = mask_near_gap(kf_df, gaps_df)  if len(kf_df)  else pd.Series(dtype=bool)
pid_df["near_gap"] = mask_near_gap(pid_df, gaps_df) if len(pid_df) else pd.Series(dtype=bool)

if len(kf_df):
    print(f"KF  ticks masked as gap-affected: {kf_df.near_gap.sum()} / {len(kf_df)} ({kf_df.near_gap.mean()*100:.1f}%)")
if len(pid_df):
    print(f"PID ticks masked as gap-affected: {pid_df.near_gap.sum()} / {len(pid_df)} ({pid_df.near_gap.mean()*100:.1f}%)")


# Startup masking (applies to every plot below, same as gap masking)

A fresh driver start has a real, one-off transient, in *both* streams: `theta_ref` isn't established until the first control cycle runs, so the earliest KFLOG tick(s) can carry raw absolute angles (seen directly in a real capture: M1/M2 `theta_meas`/`theta_state` over 100 degrees on the very first sample) instead of the near-zero reference-relative residual every later tick has, and PIDLOG's `Delta_sp` jumps by the full initial setpoint-establishment step in the first ~0.2s. Both settle within `STARTUP_EXCLUDE_SEC`. Masked the same way as `near_gap` (as `near_startup`), and combined into one `exclude` column used everywhere below -- so every plot/metric excludes it consistently instead of each cell reinventing its own threshold (this is exactly what happened before: the RA/PA anomaly hunt further down had its own local startup exclusion, but the KF plots/variation-reduction report above it didn't).

In [ ]:
STARTUP_EXCLUDE_SEC = 30.0   # seconds from the start of the run to mask as startup transient

kf_df["near_startup"]  = (kf_df.t_sec  < STARTUP_EXCLUDE_SEC) if len(kf_df)  else pd.Series(dtype=bool)
pid_df["near_startup"] = (pid_df.t_sec < STARTUP_EXCLUDE_SEC) if len(pid_df) else pd.Series(dtype=bool)

kf_df["exclude"]  = (kf_df.near_gap  | kf_df.near_startup)  if len(kf_df)  else pd.Series(dtype=bool)
pid_df["exclude"] = (pid_df.near_gap | pid_df.near_startup) if len(pid_df) else pd.Series(dtype=bool)

if len(kf_df):
    print(f"KF  ticks masked as startup transient: {kf_df.near_startup.sum()} / {len(kf_df)} "
          f"({kf_df.near_startup.mean()*100:.1f}%)")
if len(pid_df):
    print(f"PID ticks masked as startup transient: {pid_df.near_startup.sum()} / {len(pid_df)} "
          f"({pid_df.near_startup.mean()*100:.1f}%)")


# KF: Measured vs Filtered, position already reference-relative, velocity vs ω_ref

`θ_meas`/`θ_state` already have `theta_ref` subtracted by the driver whenever tracking is
enabled (`control.py`'s `observe()` passes `theta_ref=self._pid.theta_ref` into
`wrap_angle_residual()`) -- plotting them directly, with no further subtraction, is correct.
`ω_meas`/`ω_state` are different: KFLOG logs those completely raw
(`omega_meas.flatten().tolist()`, never passed through `wrap_angle_residual`), so comparing
them to `ω_ref` (also a KFLOG field) has to happen here in the notebook, not in the driver.

In [ ]:
fig = make_subplots(rows=3, cols=2, shared_xaxes=True,
    subplot_titles=sum([[f"{ax} -- θ_meas vs θ_state (arcsec, already ref-relative)", f"{ax} -- (ω_meas - ω_ref) vs (ω_state - ω_ref) (arcsec/s)"] for ax in AXES], []),
    vertical_spacing=0.06)
exclude = kf_df.exclude
for i, ax in enumerate(AXES):
    row = i + 1
    meas_err  = (kf_df[f"θ_meas_{row}"]  * ARCSEC).mask(exclude)
    state_err = (kf_df[f"θ_state_{row}"] * ARCSEC).mask(exclude)
    fig.add_trace(go.Scatter(x=kf_df.t_sec, y=meas_err, name=f"{ax} θ_meas",
        line=dict(color="royalblue", width=1)), row=row, col=1)
    fig.add_trace(go.Scatter(x=kf_df.t_sec, y=state_err, name=f"{ax} θ_state",
        line=dict(color="white", width=1)), row=row, col=1)
    meas_verr  = ((kf_df[f"ω_meas_{row}"]  - kf_df[f"ω_ref_{row}"]) * ARCSEC).mask(exclude)
    state_verr = ((kf_df[f"ω_state_{row}"] - kf_df[f"ω_ref_{row}"]) * ARCSEC).mask(exclude)
    fig.add_trace(go.Scatter(x=kf_df.t_sec, y=meas_verr, name=f"{ax} ω_meas-ω_ref",
        line=dict(color="royalblue", width=1)), row=row, col=2)
    fig.add_trace(go.Scatter(x=kf_df.t_sec, y=state_verr, name=f"{ax} ω_state-ω_ref",
        line=dict(color="white", width=1)), row=row, col=2)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=2)
fig.update_layout(height=900, width=1500, template="plotly_dark", hovermode="x unified",
    title="Kalman Filter -- Measured vs Filtered, relative to reference", legend=dict(groupclick="toggleitem"))
fig.show()


# KF: variation reduced vs the raw input signal

Reports `std(θ_state)` against `std(θ_meas)` per axis (and the equivalent against `ω_ref`
for `ω`, see markdown above) as a direct "how much noise did the filter remove" number --
gap-affected ticks excluded, since a few huge dropout-recovery spikes would otherwise
dominate a std() and make the filter look far more/less effective than it actually is on
ordinary ticks.

A large std here doesn't necessarily mean noise, though -- it's just as consistent with a
real, systematic bias or drift (constant offset, or a steady trend over the session) sitting
underneath. Since std alone can't tell those apart, this also reports mean and a linear
time-trend per axis: a mean far from zero with a *small* trend looks like a persistent bias;
a small mean with a large trend looks like something drifting away over the session (e.g. an
uncorrected/mis-signed periodic or secular term); large std with both small is closer to
genuine noise.

In [ ]:
clean_kf = kf_df[~kf_df.exclude] if len(kf_df) else kf_df

SANE_STD_ARCSEC = 30.0   # flag axes whose std is this large -- worth characterizing further below

def report_reduction(label, unit, meas, state):
    raw_std  = meas.std()
    filt_std = state.std()
    pct = (1 - filt_std / raw_std) * 100 if raw_std > 0 else float("nan")
    word = "reduction" if pct >= 0 else "increase"
    flag = f"  <-- std > {SANE_STD_ARCSEC:.0f}, see characterization below" if raw_std > SANE_STD_ARCSEC else ""
    print(f"  {label}: raw std={raw_std:9.3f}{unit} -> filtered std={filt_std:9.3f}{unit}  ({pct:+.1f}% {word}){flag}")

def characterize(label, series, t_sec):
    slope = np.polyfit(t_sec, series, 1)[0] * 3600  # arcsec/hr
    half = t_sec.min() + (t_sec.max() - t_sec.min()) / 2
    first_half_mean  = series[t_sec < half].mean()
    second_half_mean = series[t_sec >= half].mean()
    print(f"    {label}: mean={series.mean():+9.2f}\"  trend={slope:+9.2f}\"/hr  "
          f"1st-half mean={first_half_mean:+9.2f}\"  2nd-half mean={second_half_mean:+9.2f}\"")

if len(clean_kf):
    print("Position (θ, arcsec):")
    large_axes = []
    for i, ax in enumerate(AXES):
        meas  = clean_kf[f"θ_meas_{i+1}"]  * ARCSEC
        state = clean_kf[f"θ_state_{i+1}"] * ARCSEC
        report_reduction(ax, '"', meas, state)
        if meas.std() > SANE_STD_ARCSEC:
            large_axes.append((ax, meas))

    print("\nVelocity (ω - ω_ref, arcsec/s):")
    for i, ax in enumerate(AXES):
        meas_verr  = (clean_kf[f"ω_meas_{i+1}"]  - clean_kf[f"ω_ref_{i+1}"]) * ARCSEC
        state_verr = (clean_kf[f"ω_state_{i+1}"] - clean_kf[f"ω_ref_{i+1}"]) * ARCSEC
        report_reduction(ax, '"/s', meas_verr, state_verr)

    if large_axes:
        print(f"\nCharacterizing axes with std > {SANE_STD_ARCSEC:.0f}\" (bias vs. drift vs. noise):")
        for ax, meas in large_axes:
            characterize(ax, meas, clean_kf.t_sec)
else:
    print("No clean KF data to compute a reduction metric from.")

# KF: Gain per axis (position + velocity)

`K_gain` is the diagonal of the Kalman gain matrix: indices 1-3 are the position gains
(θ1-3), 4-6 the velocity gains (ω1-3). A gain spike means the filter suddenly started
trusting a raw measurement much more than usual -- worth cross-checking against any
θ_meas/θ_state divergence at the same tick.

In [ ]:
fig = go.Figure()
labels = [f"{ax} pos" for ax in AXES] + [f"{ax} vel" for ax in AXES]
colors = ["royalblue", "orange", "mediumseagreen", "royalblue", "orange", "mediumseagreen"]
dashes = ["solid", "solid", "solid", "dash", "dash", "dash"]
for i, (label, color, dash) in enumerate(zip(labels, colors, dashes)):
    fig.add_trace(go.Scatter(x=kf_df.t_sec, y=kf_df[f"K_gain_{i+1}"].mask(kf_df.exclude), name=label,
        line=dict(color=color, width=1, dash=dash)))
fig.update_layout(height=500, width=1300, template="plotly_dark", hovermode="x unified",
    title="Kalman Gain per axis", xaxis_title="Time (s)", yaxis_title="Gain",
    legend=dict(groupclick="toggleitem"))
fig.show()


# PID: Position tracking error (θ, Base-frame motor angles)

Plotted as `θ_pv - θ_sp` (arcsec, wrap-safe -- see `wrap_deg`) rather than the two raw lines
overlaid -- M1-M3 drift together as the mount tracks, so an SP-vs-PV overlay is two
near-identical slowly-moving lines with the actual tracking error invisible at that scale.
This is the error directly.

In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=[f"{ax} -- θ_pv - θ_sp (arcsec)" for ax in AXES], vertical_spacing=0.08)
for i, ax in enumerate(AXES):
    row = i + 1
    error = (wrap_deg(pid_df[f"θ_pv_{row}"] - pid_df[f"θ_sp_{row}"]) * ARCSEC).mask(pid_df.exclude)
    fig.add_trace(go.Scatter(x=pid_df.t_sec, y=error, name=f"{ax} error",
        line=dict(color="white", width=1)), row=row, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.update_layout(height=800, width=1300, template="plotly_dark", hovermode="x unified",
    title="PID Position Tracking Error", legend=dict(groupclick="toggleitem"))
fig.show()


# PID: Velocity breakdown (Kp / Ki / Kd / FF / Output)

Matches the Alpaca Pilot PID Tuning page convention: Cyan = Output (ω_op, what actually
drove the motor), Magenta = Kp, Olive = Ki, Orange = Kd, Green = FF (currently ω_ff − ω_pec
combined -- PEC is not yet broken out as its own field in the payload).

In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
    subplot_titles=[f"{ax} -- velocity components (arcsec/s)" for ax in AXES], vertical_spacing=0.08)
components = [("ω_ff", "green"), ("ω_kp", "magenta"), ("ω_ki", "olive"), ("ω_kd", "orange"), ("ω_op", "cyan")]
for i, ax in enumerate(AXES):
    row = i + 1
    for key, color in components:
        fig.add_trace(go.Scatter(x=pid_df.t_sec, y=(pid_df[f"{key}_{row}"] * ARCSEC).mask(pid_df.exclude), name=f"{ax} {key}",
            line=dict(color=color, width=1)), row=row, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.update_layout(height=900, width=1300, template="plotly_dark", hovermode="x unified",
    title="PID Velocity Breakdown", legend=dict(groupclick="toggleitem"))
fig.show()


# PID: Equatorial & Topocentric tracking error (RA/Dec/PA, Az/Alt/Roll)

Issue #88's reported symptom (near-meridian RA/PA noise) shows up in the **equatorial**
frame (`Δ_*`), not the base motor frame (`θ_*`) plotted above -- M1/M2/M3 look clean even
during an affected period, so this view is the one that actually matters for that
investigation. Both columns plot `pv - sp` (error, wrap-safe -- see `wrap_deg`) rather than
an SP/PV overlay. RA/Dec need this because they barely move during single-target tracking, so
the raw lines would overlap and hide the error -- but Az/Alt/Roll need it too, for a different
reason: they *do* move a lot over a session (tens of degrees), so the actual tracking error
(arcsec-scale) would be completely invisible against that range on a raw overlay, and Az
crossing 0°/360° would otherwise show as a spurious ~360° spike without the wrap-safe diff.
`α_pv_1` = Az, marked against meridian transit (Az≈178-180°) with a dotted line for reference.

In [ ]:
fig = make_subplots(rows=3, cols=2, shared_xaxes=True,
    subplot_titles=sum([[f"{ax} -- Δ_pv - Δ_sp (arcsec)", f"{TOPO[i]} -- α_pv - α_sp (arcsec)"] for i, ax in enumerate(EQU)], []),
    vertical_spacing=0.06)
exclude = pid_df.exclude
for i in range(3):
    row = i + 1
    equ_error  = (wrap_deg(pid_df[f"Δ_pv_{row}"] - pid_df[f"Δ_sp_{row}"]) * ARCSEC).mask(exclude)
    topo_error = (wrap_deg(pid_df[f"α_pv_{row}"] - pid_df[f"α_sp_{row}"]) * ARCSEC).mask(exclude)
    fig.add_trace(go.Scatter(x=pid_df.t_sec, y=equ_error, name=f"{EQU[i]} error",
        line=dict(color="white", width=1)), row=row, col=1)
    fig.add_trace(go.Scatter(x=pid_df.t_sec, y=topo_error, name=f"{TOPO[i]} error",
        line=dict(color="white", width=1)), row=row, col=2)
fig.update_xaxes(title_text="Time (s)", row=3, col=1)
fig.update_xaxes(title_text="Time (s)", row=3, col=2)
fig.update_layout(height=900, width=1500, template="plotly_dark", hovermode="x unified",
    title="PID Equatorial (left) & Topocentric (right) Tracking Error",
    legend=dict(groupclick="toggleitem"))
fig.show()
# Az range / meridian-proximity is checked separately in the RA/PA anomaly-hunt section below,
# since absolute Az isn't on this chart anymore now that the right column is error, not SP/PV.


# Combined KF+PID timeline

Merges the two streams on nearest timestamp (they tick independently -- KF on every 518
message, PID on every control step -- so this aligns them within a short tolerance rather
than assuming a shared clock). Used by the "zoom into context" cells below.

In [ ]:
TOLERANCE = pd.Timedelta("100ms")

merged = pd.DataFrame()
if len(kf_df) and len(pid_df):
    merged = pd.merge_asof(
        pid_df.sort_values("timestamp"), kf_df.sort_values("timestamp"),
        on="timestamp", direction="nearest", tolerance=TOLERANCE,
        suffixes=("_pid", "_kf"),
    )
    merged["t_sec"] = (merged["timestamp"] - merged["timestamp"].iloc[0]).dt.total_seconds()
print(f"Merged samples: {len(merged)}")
merged.head()


# Anomaly Detection & Investigation

Three distinct failure signatures get their own detection + investigation pair below, instead
of one generic "PID reacted hard" bucket -- each has a different likely root cause and needs
different context to diagnose:

- **Spike Anomaly** -- a sudden, sharp PID reaction on one axis (a lone hard kick, or a short
  multi-tick "hump"). Usually the PID responding correctly to something real -- the question
  is *what* disturbed it.
- **Oscillation Anomaly** -- a sustained, periodic tracking error (multi-second period,
  survives several cycles) rather than a one-off transient. Originally reported near meridian
  transit as mirrored RA/PA error (issue #88), but detected here directly in motor-frame
  position error at **any** orientation, since a real periodic disturbance shows up there
  regardless of where on the sky the mount happens to be pointing.
- **Comms Anomaly** -- a 518 telemetry dropout. Previously only masked out of the other plots
  as `near_gap`; this section instead characterizes each dropout on its own terms -- how long,
  what state the driver was in going in and coming out, how far position had to be recovered.

Each gets the same two-part treatment:
1. **Detection** -- an automatic scan producing a summary table of discrete *events* (not
   per-tick), with tunable thresholds documented inline, same convention as the rest of this
   notebook.
2. **Anomoly Analysis** -- an interactive explorer (dropdown, or step through with the slider) showing
   the detailed telemetry leading up to, through, and recovering from each detected event, so
   a flagged event can actually be diagnosed, not just counted.

In [ ]:
# ── Shared anomaly-hunter helpers (used by all three anomaly sections below) ────────────────
AXIS_COLORS = {1: ("royalblue", "deepskyblue"), 2: ("mediumseagreen", "palegreen"), 3: ("orange", "navajowhite")}
# (meas_color, state_color) per 1-based axis index -- consistent across every hunter below so
# "which axis is which colour" doesn't have to be relearned per section.

def event_shapes(t_start, t_end, t_center):
    """Dashed line at the peak/center tick (t=0) plus a shaded band over the event's actual
    span, both expressed relative to the event center so they match the traces' x-axis."""
    return [
        dict(type="line", xref="x", yref="paper", x0=0, x1=0, y0=0, y1=1,
             line=dict(color="red", width=1, dash="dash")),
        dict(type="rect", xref="x", yref="paper",
             x0=t_start - t_center, x1=t_end - t_center, y0=0, y1=1,
             fillcolor="red", opacity=0.08, line_width=0),
    ]

def window_slice(df, t_center, before_sec, after_sec=None):
    """[t_center - before_sec, t_center + after_sec] of a KF/PID dataframe, with t_sec rebased
    to 0 at t_center so different events' traces line up for direct comparison. after_sec
    defaults to before_sec (a symmetric window) -- Comms events pass distinct before/after."""
    if after_sec is None:
        after_sec = before_sec
    w = df[(df.t_sec > t_center - before_sec) & (df.t_sec < t_center + after_sec)].copy()
    w["t_rel"] = w.t_sec - t_center
    return w

def event_window(ev, window_sec):
    """(before, after) seconds either side of ev.t_sec to display. Uses the event's own
    window_before/window_after when the events_df provides them (Oscillation: sized to the
    event's real detected span plus a few periods of context, since a fixed +/-20s can run
    deep into masked/irrelevant data for a short fast oscillation, or barely cover a long
    slow one). Falls back to the shared symmetric window_sec otherwise (Spike: a fixed
    context window makes sense there, its events are always a couple of ticks wide)."""
    has_custom = "window_before" in ev.index and "window_after" in ev.index \
                 and pd.notna(ev.window_before) and pd.notna(ev.window_after)
    if has_custom:
        return float(ev.window_before), float(ev.window_after)
    return window_sec, window_sec

def clipped_shapes(ev, before, after):
    """event_shapes(), but with t_start/t_end first clipped to the actual displayed window.
    An event's own span can run well past the hunter's window (seen with Oscillation events,
    whose t_start/t_end come from a sparse detection grid -- Spike events never do, their
    span is a couple of ticks, which is why this only surfaced there). Plotly's autorange
    includes shapes as well as traces, so an unclipped rectangle reaching past the real data
    silently stretches the axis out to fit it, squeezing the actual traces into a fraction of
    the plot and making a real oscillation look artificially flat."""
    t_start = max(ev.t_start, ev.t_sec - before)
    t_end = min(ev.t_end, ev.t_sec + after)
    return event_shapes(t_start, t_end, ev.t_sec)

def single_axis_event_hunter(events_df, window_sec, title_prefix):
    """Interactive hunter for single-axis events (Spike, Oscillation): KF/PID context around
    each flagged event (see event_window() for how wide), one dropdown/slider entry per
    event, 4 panels (518 timing, theta position deviation, theta error, velocity components)
    for the one axis involved. Shared by both anomaly types below -- same navigation
    mechanism, just a different events_df and title."""
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
        specs=[[{}], [{"secondary_y": True}], [{}], [{}]],
        subplot_titles=["518 timing: receipt interval vs measurement_lag_s (s)",
                         "\u03b8_meas_dev / \u03b8_state_dev (arcsec) -- \u03b8_meas_true / \u03b8_state_true hidden, own axis (deg)",
                         "\u03b8 error (arcsec)", "velocity components (arcsec/s)"],
        row_heights=[0.2, 0.3, 0.2, 0.3], vertical_spacing=0.06)

    if not len(events_df):
        print("No events flagged -- nothing to hunt.")
        return

    events = events_df.reset_index(drop=True)
    components = [("\u03c9_ff", "green"), ("\u03c9_kp", "magenta"), ("\u03c9_ki", "olive"), ("\u03c9_kd", "orange"), ("\u03c9_op", "cyan")]
    trace_spans = []
    true_trace_indices = []
    for ev_i, ev in events.iterrows():
        ax_idx = ev["axes"][0]   # bracket access -- ev.axes collides with pandas' own Series.axes property
        visible = (ev_i == 0)
        start = len(fig.data)

        before, after = event_window(ev, window_sec)
        kf_window = window_slice(kf_df, ev.t_sec, before, after) if len(kf_df) else pd.DataFrame()
        t_rel_kf = kf_window.t_rel if len(kf_window) else pd.Series(dtype=float)

        fig.add_trace(go.Scatter(x=t_rel_kf, y=(kf_window.gap_sec if len(kf_window) else []), name="518 receipt", visible=visible,
            mode="markers+lines", marker=dict(size=4, color="yellow"), line=dict(color="yellow", width=1)), row=1, col=1)
        lag_col = kf_window["measurement_lag_s"] if len(kf_window) and "measurement_lag_s" in kf_window.columns else pd.Series(dtype=float)
        fig.add_trace(go.Scatter(x=t_rel_kf, y=lag_col, name="measurement_lag_s", visible=visible,
            mode="markers+lines", marker=dict(size=4, color="orange"), line=dict(color="orange", width=1)), row=1, col=1)

        meas_dev  = (kf_window[f"\u03b8_meas_{ax_idx}"]  * ARCSEC).mask(kf_window.exclude) if len(kf_window) else pd.Series(dtype=float)
        state_dev = (kf_window[f"\u03b8_state_{ax_idx}"] * ARCSEC).mask(kf_window.exclude) if len(kf_window) else pd.Series(dtype=float)
        meas_color, state_color = AXIS_COLORS.get(ax_idx, ("royalblue", "white"))
        fig.add_trace(go.Scatter(x=t_rel_kf, y=meas_dev, name="\u03b8_meas_dev", visible=visible,
            line=dict(color=meas_color, width=1)), row=2, col=1, secondary_y=False)
        fig.add_trace(go.Scatter(x=t_rel_kf, y=state_dev, name="\u03b8_state_dev", visible=visible,
            line=dict(color="white", width=1)), row=2, col=1, secondary_y=False)

        meas_raw_col = f"\u03b8_meas_raw_{ax_idx}"
        ref_raw_col  = f"\u03b8_ref_raw_{ax_idx}"
        omega_ref_col = f"\u03c9_ref_{ax_idx}"
        have_raw = len(kf_window) and meas_raw_col in kf_window.columns and ref_raw_col in kf_window.columns and "measurement_lag_s" in kf_window.columns
        if have_raw:
            meas_true = kf_window[meas_raw_col].mask(kf_window.exclude)
            theta_ref_backdated = kf_window[ref_raw_col] - kf_window[omega_ref_col] * kf_window["measurement_lag_s"]
            state_true = (kf_window[f"\u03b8_state_{ax_idx}"] + theta_ref_backdated).mask(kf_window.exclude)
        else:
            meas_true = pd.Series(dtype=float)
            state_true = pd.Series(dtype=float)
        meas_true_idx = len(fig.data)
        fig.add_trace(go.Scatter(x=t_rel_kf, y=meas_true, name="\u03b8_meas_true", visible=False,
            line=dict(color="deepskyblue", width=1, dash="dot")), row=2, col=1, secondary_y=True)
        state_true_idx = len(fig.data)
        fig.add_trace(go.Scatter(x=t_rel_kf, y=state_true, name="\u03b8_state_true", visible=False,
            line=dict(color="lightgray", width=1, dash="dot")), row=2, col=1, secondary_y=True)
        true_trace_indices.append((meas_true_idx, state_true_idx))

        window = window_slice(pid_df, ev.t_sec, before, after)
        t_rel = window.t_rel
        error = (wrap_deg(window[f"\u03b8_pv_{ax_idx}"] - window[f"\u03b8_sp_{ax_idx}"]) * ARCSEC).mask(window.exclude)
        fig.add_trace(go.Scatter(x=t_rel, y=error, name="\u03b8 error", visible=visible,
            line=dict(color="white", width=1)), row=3, col=1)
        for key, color in components:
            fig.add_trace(go.Scatter(x=t_rel, y=(window[f"{key}_{ax_idx}"] * ARCSEC).mask(window.exclude),
                name=key, visible=visible, line=dict(color=color, width=1)), row=4, col=1)
        trace_spans.append((start, len(fig.data) - start))

    # Toggling trace visibility via an updatemenu does NOT itself make Plotly re-fit each
    # axis to whatever's now visible -- every axis keeps whatever range it was first given
    # (autorange computed for event 0's data at initial render), so later events can appear
    # to not update at all (a smaller-magnitude row looks flat/unchanged against a scale
    # sized for a bigger one) or look truncated (a differently-shaped window against a
    # locked-in range). Forcing autorange back on for every axis on every click fixes both --
    # confirmed by inspecting a real generated figure's layout, where shared_xaxes=True had
    # already linked every row's x-axis to row 4's (xaxis.matches="x4" etc.), which is exactly
    # what makes a stale range on any one of them look consistent across all four instead of
    # being an obvious mismatch.
    AXIS_AUTORANGE = {f"{ax}.autorange": True for ax in
                       ["xaxis", "xaxis2", "xaxis3", "xaxis4", "yaxis", "yaxis2", "yaxis3", "yaxis4", "yaxis5"]}

    n_total = len(fig.data)
    buttons = []
    for ev_i, ev in events.iterrows():
        start, count = trace_spans[ev_i]
        visible = [False] * n_total
        for j in range(start, start + count):
            visible[j] = True
        for idx in true_trace_indices[ev_i]:
            visible[idx] = False
        buttons.append(dict(
            label=ev.short_label, method="update",
            args=[{"visible": visible},
                  {"title": f"{title_prefix}: {ev.label}", "shapes": clipped_shapes(ev, *event_window(ev, window_sec)),
                   **AXIS_AUTORANGE}],
        ))

    slider_steps = list(buttons)   # same rich label as the dropdown (time + description), not a bare index -- a slider step needs to say what t is on its own, since moving the slider does not update the dropdown's own highlighted selection (a real Plotly limitation: the two controls share the same click/drag handler but neither is aware of the other's displayed state)

    fig.add_hline(y=0.2, row=1, col=1, line=dict(color="gray", width=1, dash="dot"))
    fig.update_yaxes(title_text="\u03b8 absolute (deg)", row=2, col=1, secondary_y=True)
    fig.update_xaxes(title_text="Time relative to event peak (s)", row=4, col=1)
    fig.update_layout(height=1100, width=1300, template="plotly_dark", hovermode="x unified",
        title=f"{title_prefix}: {events.iloc[0].label}",
        shapes=clipped_shapes(events.iloc[0], *event_window(events.iloc[0], window_sec)),
        legend=dict(groupclick="toggleitem"),
        updatemenus=[dict(buttons=buttons, direction="down", x=1.0, xanchor="right", y=1.1, yanchor="top",
                           showactive=True, bgcolor="#2B2B2B", bordercolor="#777777", borderwidth=1,
                           font=dict(color="#AAA"))],
        sliders=[dict(active=0, x=0.0, len=0.88, pad=dict(t=60, b=10),
                       currentvalue=dict(prefix="Event (use \u2190/\u2192 or drag to step): ", font=dict(color="#EEEEEE", size=12)),
                       font=dict(color="#EEEEEE", size=10), bgcolor="#2B2B2B", bordercolor="#777777",
                       activebgcolor="#555555", steps=slider_steps)])
    fig.show()

## Spike Anomaly: sudden PID reactions

Flags *events* -- runs of consecutive control ticks where `|\u03c9_kp|` on any axis sits far
above its local baseline -- a direct proxy for "the PID suddenly thought it was a long way
off target and kicked the motor hard", the kind of transient that could put a bend in a star
trail without ever crossing the log's `WARNING` lag thresholds. Uses a rolling median + MAD
(robust to the spikes themselves, unlike mean/stdev) for the per-tick threshold. Flagging
combines two rules so both shapes of disturbance get caught without drowning in one-off
noise: a single tick clearing the high `N_MAD_SPIKE` bar is an event on its own (a lone sharp
kick), while a lower `N_MAD_RUN` bar only counts with `MIN_RUN`+ consecutive ticks (a slower
multi-second "hump" that never spikes hard on any one tick). Consecutive flagged ticks on the
same axis within `CLUSTER_GAP_S` of each other are merged into a single event (start/end/peak)
rather than listed per-tick.

`MANUAL_SPIKE_EVENTS` below is an escape hatch for anything spotted by eye (e.g. in the
hunter) that the automatic thresholds don't clear -- add `(axis, t_sec)` and it's folded into
the same table/hunter, tagged `manual=True`, with no detection logic applied.

### Detection

In [ ]:
WINDOW = 51          # ticks (~10s at 200ms) for the rolling baseline
N_MAD_SPIKE = 8.0    # a single tick this far above baseline is an event on its own
N_MAD_RUN = 5.0      # a lower bar, but only counts with >=MIN_RUN consecutive ticks
MIN_RUN = 2          # minimum consecutive flagged ticks (same axis) for the N_MAD_RUN bar to count
CLUSTER_GAP_S = 1.5  # merge flagged ticks on the same axis into one event if within this many
                     # seconds of each other, so one multi-tick disturbance is one row, not N

MANUAL_SPIKE_EVENTS = [
#    ("M2", 1457.04),  # spotted by eye in the hunter around the M2 @ 1447.2s event -- a real
                       # ~2.4s sustained excursion, but its peak (~1.7 arcsec/s) never clears
                       # N_MAD_RUN against this stretch's low local baseline (~0.7), so automatic
                       # detection can't reach it without flooding the whole run with false positives
]

events = []
axis_stats = {}
for i, ax in enumerate(AXES):
    col = f"\u03c9_kp_{i+1}"
    series = (pid_df[col] * ARCSEC).abs()
    med = series.rolling(WINDOW, center=True, min_periods=WINDOW//2).median()
    mad = (series - med).abs().rolling(WINDOW, center=True, min_periods=WINDOW//2).median()
    robust_std = 1.4826 * mad
    axis_stats[ax] = (col, med)

    candidate = (series > med + N_MAD_RUN * robust_std) & (robust_std > 1e-6) & (~pid_df.exclude)
    flagged = pid_df.loc[candidate, ["t_sec", "timestamp"]].copy()
    flagged["val"] = pid_df.loc[candidate, col] * ARCSEC
    flagged["is_spike"] = (series[candidate] > (med[candidate] + N_MAD_SPIKE * robust_std[candidate])).values
    if not len(flagged):
        continue
    group = (flagged.t_sec.diff().fillna(0) > CLUSTER_GAP_S).cumsum()
    for _, g in flagged.assign(group=group).groupby("group"):
        if len(g) < MIN_RUN and not g.is_spike.any():
            continue
        peak = g.loc[g.val.abs().idxmax()]
        events.append(dict(axis=ax, axes=(i + 1,), t_sec=peak.t_sec, timestamp=peak.timestamp,
                            t_start=g.t_sec.min(), t_end=g.t_sec.max(), n_ticks=len(g),
                            omega_kp_arcsec_s=peak.val, local_median_arcsec_s=med.loc[peak.name],
                            manual=False,
                            short_label=f"{ax} @ {peak.t_sec:.1f}s",
                            label=f"{ax} @ {peak.t_sec:.1f}s ({peak.val:.1f} arcsec/s, {len(g)} ticks)"))

for ax, t_center in MANUAL_SPIKE_EVENTS:
    i = AXES.index(ax)
    col, med = axis_stats[ax]
    nearest = (pid_df.t_sec - t_center).abs().idxmin()
    val = pid_df[col][nearest] * ARCSEC
    events.append(dict(axis=ax, axes=(i + 1,), t_sec=pid_df.t_sec[nearest], timestamp=pid_df.timestamp[nearest],
                        t_start=pid_df.t_sec[nearest], t_end=pid_df.t_sec[nearest], n_ticks=1,
                        omega_kp_arcsec_s=val, local_median_arcsec_s=med[nearest], manual=True,
                        short_label=f"{ax} @ {t_center:.1f}s",
                        label=f"{ax} @ {t_center:.1f}s (manual)"))

spike_events_df = pd.DataFrame(events).sort_values("t_sec").reset_index(drop=True) if events else pd.DataFrame()
print(f"Flagged {len(spike_events_df)} Spike Anomaly event(s) across {len(AXES)} axes (telemetry-gap ticks excluded)")
spike_events_df.head(30)

### Anomoly Analysis
Pick any flagged Spike event from the dropdown (or step through with the slider) and see
+/-20s of position and velocity traces around it, with a shaded band marking the event's
actual start/end and a dashed marker at its peak tick (t=0). Time axis is relative to the
selected event's peak, so different events line up for direct comparison.

Four panels, top to bottom:
1. **518 timing (s)** -- raw receipt interval (yellow) alongside `measurement_lag_s` (orange;
   requires a capture with the `measurement_lag_s` KFLOG field -- older captures leave the
   panel empty).
2. **\u03b8_meas_dev / \u03b8_state_dev (arcsec)** -- deviation from a *backdated* theta_ref (see
   the KF section above), not raw position. Two more traces start **hidden** --
   `\u03b8_meas_true`/`\u03b8_state_true`, the absolute (non-backdated, own axis) values. Toggle
   via the legend to sanity-check a deviation spike against the real underlying measurement;
   they reset to hidden when you switch events.
3. **\u03b8 error (arcsec)** -- PID position error on the flagged axis.
4. **velocity components (arcsec/s)** -- \u03c9_ff/kp/ki/kd/op on the flagged axis.

In [ ]:
EVENT_WINDOW_SEC = 20.0
single_axis_event_hunter(spike_events_df, EVENT_WINDOW_SEC, "Spike Anomaly")

## Oscillation Anomaly: extended deviation from steady tracking (any orientation)

Originally reported for issue #88 as RA/PA mirrored oscillation specifically near meridian
transit (Az≈178-180°, ~4-5s period, ±3-5"), and an earlier version of this detector tried to
prove genuine periodicity directly (windowed autocorrelation, searching for a lag with a
strong self-correlation peak). That turned out fragile in practice: a smooth one-directional
ramp/transient in ω_op correlates just as well at short lags as real periodicity does, for the
unrelated reason that it isn't jagged -- confirmed against two real flagged events in this
capture that were actually settling swings, not oscillation, and each fix for one shape of
false positive (requiring zero-crossings, then detrending before testing) only shifted the
failure mode rather than closing it.

Detection here is deliberately simpler: **ω_op should stay fairly steady while tracking**, so
this flags any *sustained* stretch where a motor's short-term variance or its net rate of
change (ramp) is elevated well above its own recent robust baseline -- using the same rolling
median + MAD approach as the Spike Anomaly detector above, just applied to ω_op instead of
ω_kp. This is a strict superset of "periodic oscillation" (a real oscillation shows elevated
variance too) without needing to prove periodicity at all, so it isn't fooled by a ramp the
way the correlation-based version was -- a ramp *is* a real deviation from steady tracking,
worth flagging on its own merits, not something to filter out.

Two things keep this from just re-discovering Spike Anomaly events under a new name:
- **`OSC_MIN_DURATION_S`** requires the flagged stretch to be meaningfully longer than a spike
  ever runs -- Spike Anomaly's own `CLUSTER_GAP_S`/`MIN_RUN` are tuned for a couple of ticks
  to a couple of seconds; this requires several seconds sustained.
- Ticks already inside a flagged **Spike Anomaly** event's own span (on the same axis, with a
  little padding) are excluded outright, so the same disturbance doesn't get reported under
  both anomaly types.

Az is still recorded against every flagged event purely as *context* (was this near meridian,
where issue #88 was first seen, or somewhere else entirely) -- not as a detection gate.

**Startup exclusion:** as elsewhere in this notebook, the first `STARTUP_EXCLUDE_SEC` of a run
are dropped before flagging -- the initial setpoint-establishment jump on a fresh driver start
is a real, large one-off ω_op transient, not a tracking disturbance, and would otherwise swamp
the variance filter.

### Detection


In [ ]:
OSC_STD_WINDOW_TICKS = 25        # ~5s -- window for measuring omega_op's short-term variance/ramp
OSC_BASELINE_WINDOW_TICKS = 751  # ~150s -- window for the robust baseline "normal steady tracking" is judged against.
                                   # Must be well wider than any real disturbance is likely to run, or the baseline
                                   # itself gets contaminated by the very thing being measured against it -- confirmed
                                   # empirically: at 50s, a real ~15s ramp occupied enough of its own baseline window
                                   # that the local median tracked the ramp almost exactly, masking it entirely.
OSC_N_MAD_STD = 5.0               # local variance must clear the baseline by this many robust-MAD multiples
OSC_N_MAD_DELTA = 5.0             # local net change (ramp) must clear the baseline by this many robust-MAD multiples
OSC_MIN_DURATION_S = 3.0          # minimum sustained duration to count as Extended Deviation -- deliberately
                                   # longer than Spike Anomaly's own CLUSTER_GAP_S/MIN_RUN ever runs
OSC_CLUSTER_GAP_S = 2.0           # merge flagged runs within this many seconds of each other into one event
OSC_SPIKE_EXCLUDE_PAD_S = 1.0     # padding either side of an already-flagged Spike event's own span, when
                                   # excluding it here -- the same disturbance shouldn't get reported twice
MERIDIAN_WINDOW = 5.0             # degrees either side of Az=180 -- reported purely as event context
OSC_HUNTER_PAD_FRACTION = 0.5     # extra context shown either side of the event's own span in the hunter,
                                   # as a fraction of the event's own duration
OSC_HUNTER_MIN_PAD_S = 5.0        # floor on that padding regardless of duration

def robust_baseline(series, window):
    """Rolling median + MAD -- same robust-baseline approach the Spike Anomaly detector above
    uses, applied here to omega_op's local variance/ramp instead of |omega_kp|."""
    med = series.rolling(window, center=True, min_periods=window // 2).median()
    mad = (series - med).abs().rolling(window, center=True, min_periods=window // 2).median()
    return med, 1.4826 * mad

n = len(pid_df)
az_arr = pid_df["\u03b1_pv_1"].to_numpy() if "\u03b1_pv_1" in pid_df.columns else np.full(n, np.nan)

osc_rows = []
for i, ax in enumerate(AXES):
    series = pid_df[f"\u03c9_op_{i+1}"] * ARCSEC

    local_std = series.rolling(OSC_STD_WINDOW_TICKS, center=True, min_periods=OSC_STD_WINDOW_TICKS).std()
    std_med, std_mad = robust_baseline(local_std, OSC_BASELINE_WINDOW_TICKS)
    std_trigger = (local_std > std_med + OSC_N_MAD_STD * std_mad) & (std_mad > 1e-9)

    local_delta = series.diff(OSC_STD_WINDOW_TICKS).abs()
    delta_med, delta_mad = robust_baseline(local_delta, OSC_BASELINE_WINDOW_TICKS)
    delta_trigger = (local_delta > delta_med + OSC_N_MAD_DELTA * delta_mad) & (delta_mad > 1e-9)

    # Ticks already claimed by a Spike Anomaly event on this same axis (padded a little) don't
    # also get reported here -- see markdown above.
    spike_mask = pd.Series(False, index=pid_df.index)
    for _, sp in spike_events_df[spike_events_df.axis == ax].iterrows():
        spike_mask |= (pid_df.t_sec >= sp.t_start - OSC_SPIKE_EXCLUDE_PAD_S) & \
                      (pid_df.t_sec <= sp.t_end + OSC_SPIKE_EXCLUDE_PAD_S)

    candidate = (std_trigger | delta_trigger) & (~pid_df.exclude) & (~spike_mask)
    flagged = pid_df.loc[candidate, ["t_sec", "timestamp"]].copy()
    if not len(flagged):
        continue
    flagged["local_std"] = local_std[candidate].to_numpy()
    flagged["local_delta"] = local_delta[candidate].to_numpy()
    flagged["trigger"] = np.where(std_trigger[candidate] & delta_trigger[candidate], "variance+ramp",
                          np.where(std_trigger[candidate], "variance", "ramp"))
    flagged["az"] = az_arr[candidate.to_numpy()]

    group = (flagged.t_sec.diff().fillna(0) > OSC_CLUSTER_GAP_S).cumsum()
    for _, g in flagged.assign(group=group).groupby("group"):
        t_start, t_end = g.t_sec.min(), g.t_sec.max()
        duration = t_end - t_start
        if duration < OSC_MIN_DURATION_S:
            continue
        peak = g.loc[g.local_std.idxmax()]
        pad = max(OSC_HUNTER_PAD_FRACTION * duration, OSC_HUNTER_MIN_PAD_S)
        osc_rows.append(dict(
            axis=ax, axes=(i + 1,),
            t_sec=peak.t_sec, t_start=t_start, t_end=t_end, duration_s=duration,
            window_before=(peak.t_sec - t_start) + pad, window_after=(t_end - peak.t_sec) + pad,
            n_ticks=len(g), trigger=peak.trigger,
            peak_std_arcsec_s=peak.local_std, peak_delta_arcsec_s=peak.local_delta,
            az_min=g.az.min(), az_max=g.az.max(),
            near_meridian=bool(((g.az - 180).abs() <= MERIDIAN_WINDOW).any()),
            short_label=f"{ax} @ {peak.t_sec:.1f}s",
            label=f"{ax} @ {peak.t_sec:.1f}s ({duration:.1f}s, {peak.trigger}, "
                  f"std={peak.local_std:.1f}, \u0394={peak.local_delta:.1f} arcsec/s)",
        ))

oscillation_events_df = pd.DataFrame(osc_rows).sort_values("t_sec").reset_index(drop=True) if osc_rows else pd.DataFrame()
if len(oscillation_events_df):
    n_meridian = int(oscillation_events_df.near_meridian.sum())
    print(f"Flagged {len(oscillation_events_df)} Extended Deviation event(s) across {len(AXES)} axes "
          f"(>= {OSC_MIN_DURATION_S:.0f}s sustained, telemetry-gap and Spike-claimed ticks excluded): "
          f"{n_meridian} near Az=180\u00b1{MERIDIAN_WINDOW:.0f}\u00b0 (meridian), "
          f"{len(oscillation_events_df) - n_meridian} elsewhere.")
    display(oscillation_events_df)
else:
    print("No Oscillation Anomaly events flagged at this threshold -- try lowering OSC_N_MAD_STD / "
          "OSC_N_MAD_DELTA or OSC_MIN_DURATION_S.")


### Anomoly Analysis

Same interactive explorer as the Spike Anomaly hunter above, over the flagged Extended
Deviation events instead -- the dropdown label shows how long the deviation lasted, which
criterion tripped it (variance, ramp, or both), and the peak values for quick triage before
diving into the traces.

In [ ]:
single_axis_event_hunter(oscillation_events_df, EVENT_WINDOW_SEC, "Oscillation Anomaly")

## Comms Anomaly: 518 telemetry dropouts and sustained lag

Every plot above already excludes ticks `near_gap` (defined right after loading, from
`gaps_df`) so a dropout's KF/PID resettle transient doesn't get mistaken for a Spike or
Oscillation anomaly. This section instead looks *at* the comms problems themselves, on their
own terms, rather than just masking them out of everything else -- so it deliberately does
**not** apply the `near_gap`/`exclude` mask used elsewhere.

Two distinct sub-signatures, both surfaced in the same table below (`kind` column):

- **`dropout`** -- a hard gap, from `gaps_df` (receipt silence > `GAP_THRESHOLD_SEC`).
- **`lag`** -- 518 messages kept arriving, but consistently later than nominal, for a sustained
  run of ticks -- never long enough to register as a `gaps_df` gap, so invisible to that
  detector entirely, but still a real comms-quality problem (using the same
  `measurement_lag_s` KFLOG field the KF/PID backdating fix introduced; requires a capture
  taken after that field was added, and a run long enough for `LAG_MIN_RUN` consecutive
  elevated ticks to occur). Ticks already covered by a `dropout` are excluded here via
  `near_gap`, so a hard gap's own clamped-lag boundary samples don't get double-counted as a
  separate `lag` event.

### Detection

For each event, this pulls the driver's own state either side of it -- `age_518`/`connected`/
`mode` (if present) show whether the driver's own watchdog was already elevated *before* the
event (an early warning it was struggling) or whether it happened with no lead-up at all, and
what mode the driver was in going in and coming back. Position drift compares `\u03b8_state`
immediately before vs. immediately after -- dead reckoning during a multi-second dropout can
leave the filter's belief measurably off from where the mount actually ended up once telemetry
resumes (not computed for `lag` events -- data never actually stopped, so there's no dead
reckoning to have drifted).

In [ ]:
COMMS_LOOKBACK_S  = 5.0                 # seconds of PID history just before an event, to check for an early-warning trend
COMMS_RECOVERY_S  = GAP_EXCLUDE_AFTER   # reuse the same resettle window already used for masking (see Gap masking above)
LAG_THRESHOLD_S   = 0.15                # measurement_lag_s (s) above which a tick counts as "running late"
LAG_MIN_RUN       = 5                   # minimum consecutive elevated ticks to count as a sustained lag event --
                                         # a single late 518 on its own is normal jitter, not a comms problem
LAG_CLUSTER_GAP_S = 2.0                 # merge elevated runs within this many seconds of each other into one event

def _nearest_row(df, t_sec):
    if not len(df):
        return None
    return df.loc[(df.t_sec - t_sec).abs().idxmin()]

def _context(t_before, t_after):
    """Shared before/after driver-state lookup used by both dropout and lag events."""
    row_before = _nearest_row(pid_df, t_before)
    row_after  = _nearest_row(pid_df, t_after)
    lookback = pid_df[(pid_df.t_sec >= t_before - COMMS_LOOKBACK_S) & (pid_df.t_sec <= t_before)]
    age_518_trend = (float(lookback["age_518"].iloc[-1] - lookback["age_518"].iloc[0]))\
        if "age_518" in lookback.columns and len(lookback) >= 2 else np.nan
    return dict(
        mode_before=(row_before.get("mode") if row_before is not None else None),
        mode_after=(row_after.get("mode") if row_after is not None else None),
        age_518_before=(row_before.get("age_518") if row_before is not None else np.nan),
        age_518_trend_5s=age_518_trend,
        connected_after=(row_after.get("connected") if row_after is not None else None),
    )

# ---- Sub-detector 1: hard dropouts (from gaps_df, computed at load time) ----
comms_rows = []
theta_cols = [c for c in kf_df.columns if c.startswith("\u03b8_state_")] if len(kf_df) else []
for _, g in gaps_df.iterrows():
    t_before = g.t_sec - g.gap_sec   # last good sample's t_sec, before the gap started
    t_after  = g.t_sec               # first sample back, after the gap

    theta_before = _nearest_row(kf_df, t_before)
    theta_after  = _nearest_row(kf_df, t_after)
    if theta_cols and theta_before is not None and theta_after is not None:
        drift_arcsec = float(np.linalg.norm(
            (theta_after[theta_cols] - theta_before[theta_cols]).to_numpy(dtype=float)) * ARCSEC)
    else:
        drift_arcsec = np.nan

    t_center = (t_before + t_after) / 2   # the gap's own midpoint, not when telemetry resumed --
                                           # centering there instead skews the whole window/shape
                                           # onto one side of the actual anomaly (all of it behind
                                           # t_sec, none ahead), the more of a gap there is to show
    comms_rows.append(dict(
        kind="dropout", t_sec=t_center, t_start=t_before, t_end=t_after,   # true span only -- the
                                           # 30s recovery is context for the *window*, not the anomaly
                                           # itself, so it doesn't belong in what gets shaded
        recovery_end=t_after + COMMS_RECOVERY_S,
        gap_sec=g.gap_sec, peak_lag_s=np.nan, n_ticks=np.nan, timestamp=g.timestamp,
        drift_arcsec=drift_arcsec,
        short_label=f"dropout @ {t_center:.1f}s",
        label=f"dropout @ {t_center:.1f}s ({g.gap_sec:.1f}s)",
        **_context(t_before, t_after),
    ))

# ---- Sub-detector 2: sustained lag, never a hard enough gap to show up above ----
lag_rows = []
if len(kf_df) and "measurement_lag_s" in kf_df.columns:
    elevated = (kf_df["measurement_lag_s"] > LAG_THRESHOLD_S) & (~kf_df.near_gap)
    lag_ticks = kf_df.loc[elevated, ["t_sec", "timestamp", "measurement_lag_s"]].copy()
    if len(lag_ticks):
        group = (lag_ticks.t_sec.diff().fillna(0) > LAG_CLUSTER_GAP_S).cumsum()
        for _, run in lag_ticks.assign(group=group).groupby("group"):
            if len(run) < LAG_MIN_RUN:
                continue
            peak = run.loc[run.measurement_lag_s.idxmax()]
            t_start, t_end = run.t_sec.min(), run.t_sec.max()
            lag_rows.append(dict(
                kind="lag", t_sec=peak.t_sec, t_start=t_start, t_end=t_end,   # true span only -- see
                                           # the same note on the dropout branch above
                recovery_end=t_end + COMMS_RECOVERY_S,
                gap_sec=np.nan, peak_lag_s=peak.measurement_lag_s, n_ticks=len(run), timestamp=peak.timestamp,
                drift_arcsec=np.nan,
                short_label=f"lag @ {peak.t_sec:.1f}s",
                label=f"lag @ {peak.t_sec:.1f}s ({len(run)} ticks, peak {peak.measurement_lag_s:.2f}s)",
                **_context(t_start, t_end),
            ))
else:
    print("Note: measurement_lag_s not present in this capture's KFLOG -- lag sub-detection skipped "
          "(requires a capture taken after this field was added).")

comms_events_df = pd.DataFrame(comms_rows + lag_rows)
if len(comms_events_df):
    comms_events_df = comms_events_df.sort_values("t_sec").reset_index(drop=True)
n_dropout = int((comms_events_df.kind == "dropout").sum()) if len(comms_events_df) else 0
n_lag     = int((comms_events_df.kind == "lag").sum())     if len(comms_events_df) else 0
print(f"Characterized {len(comms_events_df)} Comms Anomaly event(s): {n_dropout} hard dropout(s) (> {GAP_THRESHOLD_SEC}s), "
      f"{n_lag} sustained-lag event(s) ({LAG_THRESHOLD_S:.2f}s+ for >= {LAG_MIN_RUN} ticks).")
if len(comms_events_df) and "age_518" not in pid_df.columns:
    print("Note: age_518/connected/mode not present in this capture's PIDLOG -- those columns will be empty. "
          "Re-capture after upgrading the driver to get the early-warning/mode context.")
comms_events_df

### Anomoly Analysis

Same event-navigation mechanism as the other two hunters, but a different layout suited to a
comms event rather than a single-axis disturbance: the window spans from before the event
through the full `COMMS_RECOVERY_S` resettle period after it (not a fixed +/-window), and
every panel below overlays all three motor axes together, since a comms problem affects the
whole system at once, not one axis. Both `dropout` and `lag` events from the table above
appear in the same dropdown/slider, distinguishable by their label prefix.

Four panels, top to bottom:
1. **518 timing (s)** -- receipt interval and `measurement_lag_s`; a dropout shows up directly
   as the interval spike, a lag event as a sustained elevated run without a spike.
2. **\u03b8_state (deg, absolute)** -- the KF's belief of position on all three axes, to see how
   far it drifted (or didn't) across the event.
3. **\u03b8 error (arcsec)** -- PID position error on all three axes, before/through/after.
4. **\u03c9_op (arcsec/s)** -- actual commanded output velocity per axis, to see how hard the PID
   corrected once telemetry resumed.

In [ ]:
COMMS_WINDOW_PAD_S = 5.0   # extra padding either side of [t_start, t_end], for context

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
    subplot_titles=["518 timing: receipt interval vs measurement_lag_s (s)",
                     "\u03b8_state (deg, absolute) -- all axes",
                     "\u03b8 error (arcsec) -- all axes", "\u03c9_op (arcsec/s) -- all axes"],
    row_heights=[0.2, 0.3, 0.25, 0.25], vertical_spacing=0.06)

if not len(comms_events_df):
    print("No Comms Anomaly events -- nothing to hunt.")
else:
    events = comms_events_df.reset_index(drop=True)
    trace_spans = []
    for ev_i, ev in events.iterrows():
        visible = (ev_i == 0)
        start = len(fig.data)
        before = ev.t_sec - ev.t_start + COMMS_WINDOW_PAD_S
        after  = ev.recovery_end - ev.t_sec + COMMS_WINDOW_PAD_S   # window extends into the resettle
                                           # period for context; the shape below doesn't, see cell 37

        kf_window = window_slice(kf_df, ev.t_sec, before, after) if len(kf_df) else pd.DataFrame()
        t_rel_kf = kf_window.t_rel if len(kf_window) else pd.Series(dtype=float)
        fig.add_trace(go.Scatter(x=t_rel_kf, y=(kf_window.gap_sec if len(kf_window) else []), name="518 receipt", visible=visible,
            mode="markers+lines", marker=dict(size=4, color="yellow"), line=dict(color="yellow", width=1)), row=1, col=1)
        lag_col = kf_window["measurement_lag_s"] if len(kf_window) and "measurement_lag_s" in kf_window.columns else pd.Series(dtype=float)
        fig.add_trace(go.Scatter(x=t_rel_kf, y=lag_col, name="measurement_lag_s", visible=visible,
            mode="markers+lines", marker=dict(size=4, color="orange"), line=dict(color="orange", width=1)), row=1, col=1)

        pid_window = window_slice(pid_df, ev.t_sec, before, after)
        t_rel = pid_window.t_rel
        for ax_idx in (1, 2, 3):
            meas_color, _ = AXIS_COLORS[ax_idx]
            state_col = f"\u03b8_state_{ax_idx}"
            state_deg = kf_window[state_col] if len(kf_window) and state_col in kf_window.columns else pd.Series(dtype=float)
            fig.add_trace(go.Scatter(x=t_rel_kf, y=state_deg, name=f"M{ax_idx} \u03b8_state", visible=visible,
                line=dict(color=meas_color, width=1)), row=2, col=1)

            error = (wrap_deg(pid_window[f"\u03b8_pv_{ax_idx}"] - pid_window[f"\u03b8_sp_{ax_idx}"]) * ARCSEC)
            fig.add_trace(go.Scatter(x=t_rel, y=error, name=f"M{ax_idx} error", visible=visible,
                line=dict(color=meas_color, width=1)), row=3, col=1)

            omega_op = pid_window[f"\u03c9_op_{ax_idx}"] * ARCSEC
            fig.add_trace(go.Scatter(x=t_rel, y=omega_op, name=f"M{ax_idx} \u03c9_op", visible=visible,
                line=dict(color=meas_color, width=1)), row=4, col=1)
        trace_spans.append((start, len(fig.data) - start))

    n_total = len(fig.data)
    buttons = []
    for ev_i, ev in events.iterrows():
        start, count = trace_spans[ev_i]
        visible = [False] * n_total
        for j in range(start, start + count):
            visible[j] = True
        buttons.append(dict(
            label=ev.short_label, method="update",
            args=[{"visible": visible},
                  {"title": f"Comms Anomaly: {ev.label}",
                   "shapes": clipped_shapes(ev, ev.t_sec - ev.t_start + COMMS_WINDOW_PAD_S,
                                             ev.recovery_end - ev.t_sec + COMMS_WINDOW_PAD_S)}],
        ))

    slider_steps = list(buttons)   # same rich label as the dropdown (time + description), not a bare index -- a slider step needs to say what t is on its own, since moving the slider does not update the dropdown's own highlighted selection (a real Plotly limitation: the two controls share the same click/drag handler but neither is aware of the other's displayed state)

    fig.update_xaxes(title_text="Time relative to event (s)", row=4, col=1)
    fig.update_layout(height=1100, width=1300, template="plotly_dark", hovermode="x unified",
        title=f"Comms Anomaly: {events.iloc[0].label}",
        shapes=clipped_shapes(events.iloc[0],
                               events.iloc[0].t_sec - events.iloc[0].t_start + COMMS_WINDOW_PAD_S,
                               events.iloc[0].recovery_end - events.iloc[0].t_sec + COMMS_WINDOW_PAD_S),
        legend=dict(groupclick="toggleitem"),
        updatemenus=[dict(buttons=buttons, direction="down", x=1.0, xanchor="right", y=1.1, yanchor="top",
                           showactive=True, bgcolor="#2B2B2B", bordercolor="#777777", borderwidth=1,
                           font=dict(color="#EEEEEE"))],
        sliders=[dict(active=0, x=0.0, len=0.88, pad=dict(t=60, b=10),
                       currentvalue=dict(prefix="Event (use \u2190/\u2192 or drag to step): ", font=dict(color="#EEEEEE", size=12)),
                       font=dict(color="#EEEEEE", size=10), bgcolor="#2B2B2B", bordercolor="#777777",
                       activebgcolor="#555555", steps=slider_steps)])
    fig.show()

# Notes